In [2]:
import os
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
load_dotenv()
api_key = os.getenv("API_KEY")

In [3]:
list_instructions = CommaSeparatedListOutputParser().get_format_instructions()

In [4]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [5]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', 
     "I've recently adopted a {pet}. Could you suggest three {pet} names? \n" + list_instructions)])

In [6]:
print(chat_template.messages[0].prompt.template)

I've recently adopted a {pet}. Could you suggest three {pet} names? 
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [7]:
chat = ChatOllama(
    model="gpt-oss:120b",
    temperature=0.7,
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {api_key}"
        }
    }
)

In [8]:
list_output_parser = CommaSeparatedListOutputParser()

In [9]:
chat_template_result = chat_template.invoke({'pet':'dog'})

In [11]:
chat_result = chat.invoke(chat_template_result)

In [14]:
print(chat_result.content)

Buddy, Luna, Max


In [15]:
list_output_parser.invoke(chat_result)

['Buddy', 'Luna', 'Max']

In [16]:
chain = chat_template | chat | list_output_parser

In [17]:
chain.invoke({'pet':'dog'})

['Buddy', 'Luna', 'Max']

In [18]:
chain = chat_template | chat

In [22]:
%%time
chain.invoke({'pet':'dog'})

CPU times: total: 62.5 ms
Wall time: 1.23 s


AIMessage(content='Buddy, Luna, Max', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-05T13:23:05.460481387Z', 'done': True, 'done_reason': 'stop', 'total_duration': 564234153, 'load_duration': None, 'prompt_eval_count': 109, 'prompt_eval_duration': None, 'eval_count': 41, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019e97f3-98f1-74a1-aa05-74914ebd83ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 41, 'total_tokens': 150})

In [23]:
%%time
chain.batch([{'pet':'dog'},{'pet':'cat'},{'pet':'rabbit'}])


CPU times: total: 78.1 ms
Wall time: 2.48 s


[AIMessage(content='Buddy, Luna, Max', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-05T13:24:32.487874813Z', 'done': True, 'done_reason': 'stop', 'total_duration': 606326814, 'load_duration': None, 'prompt_eval_count': 109, 'prompt_eval_duration': None, 'eval_count': 48, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019e97f4-eca6-7610-aaef-4ae04191c48c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 48, 'total_tokens': 157}),
 AIMessage(content='Milo, Luna, Oliver', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-05T13:24:33.779005966Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1042702727, 'load_duration': None, 'prompt_eval_count': 109, 'prompt_eval_duration': None, 'eval_count': 70, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'oll

In [32]:
%%time
response = chain.stream({'pet':'dragon'})

CPU times: total: 0 ns
Wall time: 0 ns


In [33]:
next(response)

AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019e97f6-9f26-7f52-bdb6-7ecc800c02f8', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [35]:
for i in response:
    print(i.content, end = '')

In [36]:
type(chat_template)

langchain_core.prompts.chat.ChatPromptTemplate